# S0.5 · R-A（数据出境安全评估路径）合规折损漏斗

**问题**：在最严格的出境路径下，从客户基数逐层折损后还剩多少可用样本？够不够验证方案？

方法：六层折损，每层给低/中/高三档——本项目没有真实业务数据，
任何点估计都会被误当成事实，因此一律以区间呈现，让不确定性显式化。

参数全部来自 `modules/m0_compliance/configs/s0_5_funnel_route_a.yaml`，
逻辑在 `modules/m0_compliance/components/funnel.py`。

> ⚠️ **全部折损率为显式假设，无任何真实业务来源。** 需求方确认或 M1 校准前不得作为结论引用。

In [1]:
import hashlib
import pathlib
import subprocess
import sys

import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
PERCENT = 100

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent   # nbconvert 的工作目录是 notebook 所在目录
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m0_compliance/configs/s0_5_funnel_route_a.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
config_hash = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()[:HASH_PREFIX_LEN]
git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()

print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + config_hash)
print("seed:", cfg["seed"])
print("git:", git_sha)
print("step:", cfg["step_id"], "| 路线:", cfg["route"], cfg["route_name"])

config: modules/m0_compliance/configs/s0_5_funnel_route_a.yaml sha256:f1953f6bdffc
seed: 42
git: 230892b
step: S0.5 | 路线: R-A 数据出境安全评估路径（PIPL 第 38 条第（一）项）


In [2]:
from modules.m0_compliance.components.funnel import attrition_rate, funnel

rows = funnel(cfg["base_population"], cfg["stages"])
table = pd.DataFrame(rows)

RESULT_PATH = REPO_ROOT / "modules/m0_compliance/results/route_A_funnel.csv"
table.to_csv(RESULT_PATH, index=False)

show = table[["stage", "rate_low", "rate_mid", "rate_high",
              "remaining_low", "remaining_mid", "remaining_high"]]
print(show.to_string(index=False))
print()
for scenario in ("low", "mid", "high"):
    print("%-5s 档合规折损率 N_eff / 基数 = %.5f%%"
          % (scenario, attrition_rate(rows, scenario) * PERCENT))

                      stage  rate_low  rate_mid  rate_high  remaining_low  remaining_mid  remaining_high
                       客户基数       NaN       NaN        NaN         200000         500000         1000000
                   南向通投资者资格      0.03      0.06       0.12           6000          30000          120000
                   同时为香港行客户      0.05      0.10       0.20            300           3000           24000
                      标识可匹配      0.50      0.70       0.85            150           2100           20400
       内地侧单独同意（PIPL 第 39 条）      0.15      0.30       0.50             22            630           10200
香港侧直接促销同意（PDPO 第 35C/35E 条）      0.20      0.40       0.60              4            252            6120
                     数据质量达标      0.80      0.90       0.95              3            226            5814

low   档合规折损率 N_eff / 基数 = 0.00150%
mid   档合规折损率 N_eff / 基数 = 0.04520%
high  档合规折损率 N_eff / 基数 = 0.58140%


In [3]:
threshold = cfg["minimum_viable_n_eff"]
final = rows[-1]
print("业务最小可行规模（S0.1）：N_eff ≥ %d 人" % threshold)
print("R-A 路径下可用样本区间：%d ~ %d 人（中位 %d）"
      % (final["remaining_low"], final["remaining_high"], final["remaining_mid"]))
print()
for scenario in ("low", "mid", "high"):
    reached = final["remaining_%s" % scenario]
    verdict = "达标" if reached >= threshold else "不达标"
    print("  %-5s 档：%8d 人 → %s（缺口 %d 人）"
          % (scenario, reached, verdict, max(threshold - reached, 0)))

业务最小可行规模（S0.1）：N_eff ≥ 30000 人
R-A 路径下可用样本区间：3 ~ 5814 人（中位 226）

  low   档：       3 人 → 不达标（缺口 29997 人）
  mid   档：     226 人 → 不达标（缺口 29774 人）
  high  档：    5814 人 → 不达标（缺口 24186 人）


In [4]:
# 反解：要让最乐观一档达到 30,000 人，客户基数需要多大？
rate_high = attrition_rate(rows, "high")
rate_mid = attrition_rate(rows, "mid")
need_high = threshold / rate_high
need_mid = threshold / rate_mid
print("按 high 档折损率反解，需要客户基数 ≈ %,.0f 人".replace(",", "") % need_high)
print("按 mid  档折损率反解，需要客户基数 ≈ %.0f 人" % need_mid)
print()
print("当前假设的客户基数上限为 %d 人（config 的 base_population.high）"
      % cfg["base_population"]["high"])
print("差距倍数：high 档 %.1f 倍，mid 档 %.1f 倍"
      % (need_high / cfg["base_population"]["high"], need_mid / cfg["base_population"]["high"]))

按 high 档折损率反解，需要客户基数 ≈ 5159959 人
按 mid  档折损率反解，需要客户基数 ≈ 66371681 人

当前假设的客户基数上限为 1000000 人（config 的 base_population.high）
差距倍数：high 档 5.2 倍，mid 档 66.4 倍


## 结论

**在本配置的假设下，R-A 路径的可用样本三档全部低于 S0.1 的业务最小可行规模 30,000 人。**

三点必须一起读，缺一会读错：

1. **这是假设下的结果，不是业务事实。** 六层折损率全部无来源，尤其「南向通投资者资格率」
   与「同时为香港行客户率」两层贡献了绝大部分折损，而这两个数最需要需求方提供。
   本 notebook 的价值在于**指出哪两个数最要紧**，而不是宣布结论。

2. **瓶颈在前两层，不在同意率。** 资格率 × 双持率在最乐观档也只有 2.4%，
   之后的匹配与同意再打折。把同意率提高一倍改变不了量级——
   **优化告知话术不是出路，扩大客户基数或放宽人群定义才是。**

3. **与出境阈值的耦合是本步最重要的发现，见 `route_A.md` 第五节。**
   简言之：可验证（≥30,000）与走轻量出境路径（敏感个人信息 <10,000）
   在本项目中**不可兼得**——这不是假设问题，是两条硬约束的算术关系。

> **触发条件预警**：框架 M0 的升级条件为「三条路线的可用样本估计**均**低于业务最小可行规模 → 立即上报，
> 项目命题需重新界定」。R-A 已不达标；R-B（S0.6）与 R-C（S0.7）的结构性折损层与本路线相同，
> 大概率同样不达标。**若三条均不达标，须按框架立即上报。**